# 2주차 · 패션 이미지 분류 (CNN) 👕👟

이번 주 미니 대회는 **딥러닝으로 옷 사진 분류하기**! 28×28 크기의 흑백 이미지(픽셀 784개)를 보고 티셔츠·바지·신발 등 **10가지 옷 종류 중 무엇인지** 맞히는 문제예요. 오늘은 이미지 인식의 표준 도구인 **CNN(합성곱 신경망)**을 직접 만들어봅니다.

**진행 방법 (꼭 읽어주세요!)**
1. 셀을 **위에서부터 하나씩** `Shift+Enter`로 실행하세요.
2. **✏️ 표시된 셀만** 바꿔보면 됩니다. 나머지는 그냥 실행!
3. 마지막 셀이 `submission.csv`를 다운로드해줘요 → 웹 **Submit 탭**에 업로드.

⚡ **시작 전 필수**: 메뉴에서 `런타임 > 런타임 유형 변경 > T4 GPU` 를 선택하세요. GPU 없이도 돌아가지만 학습이 훨씬 느려요.

- 평가 지표: **Accuracy** (10장 중 몇 장을 맞혔는지 — **높을수록 좋음**)
- 기본 코드 그대로도 약 90% 안팎이 나옵니다. 완주 먼저, 개선은 그다음! 😊

## 1. 데이터 불러오기  *(그냥 실행하세요)*

이미지가 CSV로 저장돼 있어요 — 한 행이 사진 한 장이고, `pixel0`~`pixel783` 컬럼이 픽셀 밝기(0~255)입니다. 이 셀은:
1. train/test CSV를 읽고
2. 픽셀들을 `28×28` 이미지 모양으로 다시 접고 (`reshape`)
3. 255로 나눠 밝기를 0~1 사이로 맞춥니다 (**정규화** — 신경망이 작은 숫자를 좋아해서 학습이 잘 돼요)

실행 후 `train: (60000, 28, 28, 1)` 같은 모양이 찍히면 성공!

In [ ]:
import pandas as pd, numpy as np  # 표 처리 + 수치 계산 라이브러리

# 데이터 공개 링크 (운영진이 채워둔 것 — 수정하지 마세요!)
TRAIN_URL = "https://raw.githubusercontent.com/nepersoned/dat-datasets/main/image/train.csv.gz"
TEST_URL  = "https://raw.githubusercontent.com/nepersoned/dat-datasets/main/image/test.csv"

if TRAIN_URL:
    train = pd.read_csv(TRAIN_URL); test = pd.read_csv(TEST_URL)  # URL에서 바로 읽기
else:
    # (예비용) URL이 없을 때만 직접 업로드 — 보통 실행되지 않아요
    from google.colab import files; files.upload()
    train = pd.read_csv("train.csv"); test = pd.read_csv("test.csv")

# 이름이 'pixel'로 시작하는 컬럼 784개를 골라냅니다
pixel_cols = [c for c in train.columns if c.startswith("pixel")]

# reshape(-1, 28, 28, 1): 픽셀 784개를 28x28 이미지 모양으로 접기 (마지막 1 = 흑백 채널)
# / 255.0: 밝기(0~255)를 0~1 사이로 정규화 → 신경망 학습이 안정적
X = train[pixel_cols].values.reshape(-1, 28, 28, 1) / 255.0
y = train["target"].values  # 정답 라벨 (0~9, 옷 종류 번호)
X_test = test[pixel_cols].values.reshape(-1, 28, 28, 1) / 255.0  # 예측 대상 이미지들

print("train:", X.shape, "| test:", X_test.shape)

## 2. 이미지 살펴보기  *(그냥 실행하세요)*

숫자 덩어리가 진짜 옷 사진인지 눈으로 확인! 앞의 6장을 그림으로 보여줍니다. 각 그림 위의 `label` 숫자가 옷 종류(0=티셔츠, 1=바지, … 9=앵클부츠)예요. 작고 흐릿해도 사람 눈에 대충 보이면, 컴퓨터도 배울 수 있습니다.

In [ ]:
import matplotlib.pyplot as plt  # 그래프/이미지 표시 라이브러리

# 1행 6열 그림판을 만들고, 학습 데이터의 앞 6장을 흑백으로 표시
fig, ax = plt.subplots(1, 6, figsize=(12, 2.5))
for i in range(6):
    ax[i].imshow(X[i].reshape(28, 28), cmap="gray")  # i번째 이미지 그리기
    ax[i].set_title(f"label {y[i]}")                  # 정답 번호를 제목으로
    ax[i].axis("off")                                 # 눈금 숨기기
plt.show()

## 3. ✏️ 여기를 바꿔보세요 — CNN 모델 만들기 & 학습

**CNN**은 이미지를 작은 창(필터)으로 훑으며 "모서리 → 무늬 → 옷 형태" 순으로 특징을 스스로 배우는 신경망이에요. 아래 구조를 읽어보세요:
- `Conv2D` = 특징 찾는 층, `MaxPooling2D` = 이미지를 절반으로 요약, `Dense` = 마지막 판단, `Dropout` = 과하게 외우는 것(과적합) 방지, `softmax` = 10개 클래스별 확률 출력

실행하면 epoch(전체 데이터 1바퀴)마다 `accuracy`가 올라가는 로그가 보여요. 5 epoch에 **val_accuracy 0.90 안팎**이면 정상이고, 몇 분 걸립니다 (GPU 켰는지 확인!).

**점수(Accuracy) 올리는 실험 아이디어** — 하나씩 바꿔서 제출해보세요:
- `epochs=5` → `10`이나 `15`로 늘리기 (가장 쉬운 개선! 단, 너무 늘리면 val_accuracy가 멈추거나 떨어져요)
- 필터 수 늘리기: `Conv2D(32, ...)` → `64`, `Conv2D(64, ...)` → `128`
- `Conv2D` + `MaxPooling2D` 쌍을 한 층 더 추가
- `Dropout(0.3)` 값을 0.2~0.5 사이에서 조절

In [ ]:
from tensorflow import keras          # 딥러닝 라이브러리 (코랩에 이미 설치돼 있어요)
from tensorflow.keras import layers   # 신경망의 '층'들

# 층을 차곡차곡 쌓아 모델을 만듭니다 (위에서 아래로 데이터가 흘러가요)
model = keras.Sequential([
    layers.Input((28, 28, 1)),                        # 입력: 28x28 흑백 이미지
    layers.Conv2D(32, 3, activation="relu"),          # 특징 찾기 1 (필터 32개)  ✏️ 64로 늘려보기
    layers.MaxPooling2D(),                            # 이미지 크기를 절반으로 요약
    layers.Conv2D(64, 3, activation="relu"),          # 특징 찾기 2 (필터 64개)  ✏️ 128로 늘려보기
    layers.MaxPooling2D(),                            # 다시 절반으로 요약
    layers.Flatten(),                                 # 2차원 이미지를 1줄로 펴기
    layers.Dense(128, activation="relu"),             # 종합 판단하는 층
    layers.Dropout(0.3),                              # 과적합 방지 (뉴런 30%를 랜덤으로 끔)
    layers.Dense(10, activation="softmax"),           # 출력: 10개 클래스 각각의 확률
])

# 컴파일: 어떻게 배울지 설정 (adam=학습 방법, loss=틀린 정도의 기준, 지표=정확도)
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

# 학습 시작! validation_split=0.1: 데이터의 10%는 시험용으로 떼어 실력(val_accuracy)을 확인
# ✏️ 여기를 바꿔보세요! epochs=5 → 10, 15 로 늘리면 보통 정확도가 올라갑니다
model.fit(X, y, epochs=5, batch_size=128, validation_split=0.1)

## 4. 예측 & 제출 파일 저장  *(실행하면 자동 다운로드)*

test 이미지 각각에 대해 모델이 내놓은 10개 확률 중 **가장 높은 클래스**(`argmax`)를 답으로 고르고, `submission.csv`로 저장합니다. 실행하면 파일이 **자동으로 다운로드**돼요.

👉 다운로드된 `submission.csv`를 웹사이트의 **Submit 탭**에 업로드하세요. 제출은 여러 번 가능하니, 모델을 바꿔가며 점수를 비교해보세요!

In [ ]:
# model.predict: 이미지마다 10개 클래스의 확률을 계산
# .argmax(axis=1): 그중 확률이 가장 높은 클래스 번호를 답으로 선택
test["prediction"] = model.predict(X_test).argmax(axis=1)

# 제출 규칙: id, prediction 두 컬럼만 저장 (index=False: 행 번호 제외)
test[["id", "prediction"]].to_csv("submission.csv", index=False)
print("저장 완료")

try:
    # 코랩에서 실행 중이면 자동 다운로드
    from google.colab import files; files.download("submission.csv")
except Exception:
    print("왼쪽 파일탭에서 submission.csv를 내려받으세요.")

## 🆘 막히면 여기를 보세요

| 증상 | 해결 |
|---|---|
| 학습이 너무 느림 (epoch 하나에 몇 분) | GPU가 꺼져 있어요 → `런타임 > 런타임 유형 변경 > T4 GPU` 선택 후 맨 위부터 재실행. |
| `NameError: name 'X' is not defined` | 위쪽 셀을 건너뛰었어요. 맨 위부터 순서대로 실행하세요. |
| 데이터 로드가 오래 걸림 | train 파일이 커서 1~2분 걸릴 수 있어요. 기다려주세요. |
| accuracy가 0.1 근처에서 안 오름 | 드문 경우인데, `런타임 > 세션 다시 시작` 후 재실행해보세요. |
| GPU 사용량 초과 경고 | 코랩 무료 GPU는 한도가 있어요. 잠시 후 다시 시도하거나 CPU로도 (느리지만) 됩니다. |

첫 딥러닝 모델 완성을 축하해요! 🎉